# EIA Electricity Demand — Exploratory Data Analysis

**Phase 2.** Run *after* `scripts/validate_raw_data.py` passes.

This notebook is for **understanding**, not for fixing. If you find a data problem
here, the fix belongs in `src/`, not in a cell — otherwise it will not exist when
Airflow runs the pipeline in Phase 8.

## 1. Setup

`load_modeling_frame()` reads the raw Parquet store and returns a filtered view.
The raw store on disk is **never modified** — narrowing the window is just a
different read, so widening it later costs nothing and needs no re-fetch.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make `src` importable from inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.validate import load_modeling_frame, load_modeling_respondents

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "eia"
REPORT  = PROJECT_ROOT / "data" / "raw" / "_logs" / "validation_report.json"

# The analysis window. Raw data starts 2019 - this is the team's chosen slice.
START = "2022-01-01"
END   = None

## 2. Loading the data

The 51 approved regions come from the validation report, not a hardcoded list,
so the notebook and the pipeline can never disagree about the modeling set.

In [ ]:
MODELING_RESPONDENTS = load_modeling_respondents(REPORT)
print(f"modeling set: {len(MODELING_RESPONDENTS)} balancing authorities")
print(MODELING_RESPONDENTS)

In [ ]:
# wide=True  -> one row per (period, respondent), series as columns  [modelling shape]
# wide=False -> one row per observation                             [raw shape]
df = load_modeling_frame(
    RAW_DIR,
    respondents=MODELING_RESPONDENTS,
    start=START,
    end=END,
    wide=True,
)

print(f"shape       : {df.shape}")
print(f"date range  : {df.period.min().date()} -> {df.period.max().date()}")
print(f"regions     : {df.respondent.nunique()}")
df.head()

### Reading the Parquet directly

`load_modeling_frame` is the recommended path, but if you want the raw long
format yourself, these are the equivalents:

```python
# One year partition
df = pd.read_parquet("data/raw/eia/year=2024/data.parquet")

# The whole store (pandas reads a partitioned directory natively)
df = pd.read_parquet("data/raw/eia")

# Only the columns you need - Parquet is columnar, so this genuinely
# avoids reading the rest from disk
df = pd.read_parquet("data/raw/eia", columns=["period", "respondent", "type", "value"])

# Push the filter down to the reader instead of loading then filtering
df = pd.read_parquet(
    "data/raw/eia",
    filters=[("type", "==", "D")],
)
```

## 3. Dataset overview

In [ ]:
print("=== dtypes ===")
print(df.dtypes.to_string())
print("\n=== missing values ===")
miss = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct": (df.isna().sum() / len(df) * 100).round(3),
})
print(miss.to_string())
print("\n=== duplicate (period, respondent) keys ===")
print(df.duplicated(["period", "respondent"]).sum())

In [ ]:
df[["demand", "demand_forecast", "net_generation", "interchange"]].describe().T

## 4. Demand distribution

Regions differ in size by orders of magnitude, so the raw distribution is
dominated by the largest few. A log scale is the honest way to see the shape.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df.demand.dropna(), bins=80, color="steelblue", edgecolor="white", linewidth=0.4)
axes[0].set_title("Daily demand (all regions)")
axes[0].set_xlabel("MWh")

axes[1].hist(df.demand.dropna(), bins=80, color="steelblue", edgecolor="white", linewidth=0.4)
axes[1].set_xscale("log")
axes[1].set_title("Daily demand — log scale")
axes[1].set_xlabel("MWh (log)")

plt.tight_layout()
plt.show()

In [ ]:
size = (df.groupby("respondent").demand.mean()
          .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(13, 5))
size.plot.bar(ax=ax, color="steelblue")
ax.set_yscale("log")
ax.set_title("Mean daily demand by region (log scale)")
ax.set_ylabel("MWh (log)")
plt.tight_layout()
plt.show()

print("largest 5:\n", size.head().round(0).to_string())
print("\nsmallest 5:\n", size.tail().round(0).to_string())
print(f"\nratio largest/smallest: {size.iloc[0] / size.iloc[-1]:,.0f}x")

## 5. Time series behaviour

Pick a few large, complete regions and look at the actual signal.

In [ ]:
FOCUS = ["PJM", "MISO", "ERCO", "CISO"]
FOCUS = [r for r in FOCUS if r in set(df.respondent)]

fig, axes = plt.subplots(len(FOCUS), 1, figsize=(13, 2.6 * len(FOCUS)), sharex=True)
for ax, r in zip(axes, FOCUS):
    s = df[df.respondent == r].set_index("period").demand
    ax.plot(s.index, s.values, linewidth=0.6, color="steelblue")
    ax.set_title(f"{r} — daily demand", loc="left", fontsize=10)
    ax.set_ylabel("MWh")
plt.tight_layout()
plt.show()

### Annual seasonality

This is the dominant pattern in electricity demand and the main reason the
analysis window matters: each additional year is one more summer/winter cycle
the model can learn from.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
for r in FOCUS:
    s = df[df.respondent == r].copy()
    monthly = s.groupby(s.period.dt.month).demand.mean()
    ax.plot(monthly.index, monthly / monthly.mean(), marker="o", label=r)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Monthly seasonality (normalised to each region's own mean)")
ax.set_ylabel("relative demand")
ax.legend()
plt.tight_layout()
plt.show()

### Weekly seasonality

Weekday/weekend structure is why `lag_7` matters for daily data.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
for r in FOCUS:
    s = df[df.respondent == r].copy()
    dow = s.groupby(s.period.dt.dayofweek).demand.mean()
    ax.plot(dow.index, dow / dow.mean(), marker="o", label=r)

ax.set_xticks(range(7))
ax.set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Day-of-week seasonality (normalised)")
ax.set_ylabel("relative demand")
ax.legend()
plt.tight_layout()
plt.show()

## 6. The benchmark: EIA's own day-ahead forecast

`demand_forecast` (DF) is EIA's published next-day prediction. It is the bar your
models must clear — beating naive persistence is easy, beating this is the
result worth reporting.

**It is a benchmark, never an input feature.**

In [ ]:
import numpy as np

b = df.dropna(subset=["demand", "demand_forecast"]).copy()
b["abs_err"] = (b.demand - b.demand_forecast).abs()
b["ape"] = b.abs_err / b.demand.replace(0, np.nan) * 100

print(f"rows compared : {len(b):,}")
print(f"MAE           : {b.abs_err.mean():>12,.0f} MWh")
print(f"RMSE          : {np.sqrt(((b.demand - b.demand_forecast) ** 2).mean()):>12,.0f} MWh")
print(f"MAPE          : {b.ape.mean():>12.2f} %")

per_region = (b.groupby("respondent")
                .agg(mape=("ape", "mean"), n=("ape", "size"))
                .sort_values("mape"))
print("\nEasiest 5 regions to forecast:\n", per_region.head().round(2).to_string())
print("\nHardest 5 regions:\n", per_region.tail().round(2).to_string())

## 7. Naive baseline

`prediction = yesterday's demand`. Any model that cannot beat this is not
learning anything.

In [ ]:
n = df.sort_values(["respondent", "period"]).copy()
n["naive"] = n.groupby("respondent").demand.shift(1)
n = n.dropna(subset=["demand", "naive"])

naive_mape = ((n.demand - n.naive).abs() / n.demand.replace(0, np.nan) * 100).mean()
print(f"naive persistence MAPE : {naive_mape:.2f} %")
print(f"EIA day-ahead    MAPE : {b.ape.mean():.2f} %")
print()
print("Targets for your models, in increasing order of difficulty:")
print(f"  1. beat naive persistence : MAPE < {naive_mape:.2f}%")
print(f"  2. beat EIA's forecast    : MAPE < {b.ape.mean():.2f}%")

## 8. Coverage and gaps

Interior gaps matter because lag features propagate NaN across them. Phase 3
must handle these explicitly.

In [ ]:
cov = (df.groupby("respondent")
         .agg(days=("period", "nunique"),
              first=("period", "min"),
              last=("period", "max"),
              missing_demand=("demand", lambda s: int(s.isna().sum()))))
span = (df.period.max() - df.period.min()).days + 1
cov["coverage_pct"] = (cov.days / span * 100).round(2)
cov["interior_gaps"] = ((cov["last"] - cov["first"]).dt.days + 1 - cov.days).clip(lower=0)

print(f"expected days in window: {span}")
print("\nregions with interior gaps or missing demand:")
issues = cov[(cov.interior_gaps > 0) | (cov.missing_demand > 0)].sort_values("interior_gaps", ascending=False)
print(issues.to_string() if len(issues) else "  none")

## 9. Proposed chronological split

**Never use a random split on time series.** Sorting by date and cutting is the
only way to avoid training on the future.

In [ ]:
dates = pd.Series(sorted(df.period.unique()))
train_end = dates.quantile(0.70, interpolation="nearest")
val_end   = dates.quantile(0.85, interpolation="nearest")

train = df[df.period <= train_end]
val   = df[(df.period > train_end) & (df.period <= val_end)]
test  = df[df.period > val_end]

for name, part in [("TRAIN", train), ("VAL", val), ("TEST", test)]:
    print(f"{name:6s} {part.period.min().date()} .. {part.period.max().date()}  "
          f"{len(part):>7,} rows  ({len(part)/len(df)*100:4.1f}%)")

assert train.period.max() < val.period.min() < test.period.min(), "temporal leakage!"
print("\nno temporal overlap between splits")

## 10. Findings → Phase 3

Record conclusions here so feature engineering has a written brief:

- **Seasonality**: annual (summer/winter peaks) and weekly (weekday/weekend).
  → justifies `lag_1`, `lag_7`, `lag_365`, and rolling means.
- **Scale**: regions differ by orders of magnitude.
  → either model per-region, or include region as a categorical feature.
- **Benchmark**: EIA's day-ahead MAPE is the number to beat.
- **Gaps**: listed in section 8; must be handled before lags are computed.
- **Split**: chronological, verified non-overlapping above.